# Properties

A *property* is an attribute whose reading, writing and deletion go through methods, while keeping the syntax of a simple attribute (no parentheses).

This lab starts from client code that must not change, then moves on to how properties work. Each exercise provides the class it starts from: you can do them in any order, and the last one is optional.

## Validating Without Changing the Client Code

The `Account` class below exposes a public `balance` attribute, and the cell after it stands for client code, written long ago and used everywhere.

New requirement: a balance can no longer be negative. Change `Account` so that any write of a negative balance raises a `ValueError`, at construction as well as afterwards, without changing a single line of the client code.

Once the class is changed, run the client code again: it must print `150`, two error messages, then `150` again.

In [ ]:
class Account:
  def __init__(self, balance):
    self.balance = balance

  # Your code here

In [ ]:
account = Account(100)
account.balance += 50
print(account.balance)

try:
  account.balance = -1
except ValueError as error:
  print(f"Error: {error}")

try:
  Account(-1)
except ValueError as error:
  print(f"Error: {error}")

print(account.balance)

### Additional hints

- `balance` becomes a property: a getter under `@property`, a setter under `@balance.setter`, both named `balance`.
- The value goes in an internal attribute with another name, `_balance`. Writing `self.balance` in the setter would call it again endlessly.
- `__init__` doesn't change: `self.balance = balance` already goes through the setter.

```python
class Account:
  def __init__(self, balance):
    self.balance = balance

  @property
  def balance(self):
    return ...

  @balance.setter
  def balance(self, value):
    if ...:
      raise ValueError("The balance can't be negative")
    self._balance = ...
```

### Solution

In [ ]:
class Account:
  def __init__(self, balance):
    self.balance = balance

  @property
  def balance(self):
    return self._balance

  @balance.setter
  def balance(self, value):
    if value < 0:
      raise ValueError("The balance can't be negative")
    self._balance = value


account = Account(100)
account.balance += 50
print(account.balance)

try:
  account.balance = -1
except ValueError as error:
  print(f"Error: {error}")

try:
  Account(-1)
except ValueError as error:
  print(f"Error: {error}")

print(account.balance)

## Read-Only Property

Add to the `Circle` class an `area` property, read-only (no setter), which computes the area of the disk from `radius` (use `math.pi`). `Circle(5).area` must be about `78.54`.

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("The radius must be strictly positive")
    self._radius = value

  # Your code here

What happens if you run `c.area = 10`? Try it before reading the solution.

*Your answer*

### Additional hints

A method under `@property`, with no `@area.setter`. It reads `self.radius` and returns $\pi r^2$.

```python
@property
def area(self):
  return ...
```

### Solution

A property with no setter is read-only: any attempt to assign to it raises an `AttributeError`.

Internally, `property` is a *data descriptor*: it defines `__set__`, even with no setter (that `__set__` then raises the error), which gives it priority over the instance's `__dict__`.

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("The radius must be strictly positive")
    self._radius = value

  @property
  def area(self):
    return math.pi * self.radius**2


c = Circle(5)
print(c.area)

try:
  c.area = 10
except AttributeError as error:
  print(f"Error: {error}")

## Invalidating a Cache

Computing `area` is trivial here, but let's imagine that it takes long, like a numerical integration on a very fine grid. The class below caches it with `functools.cached_property`.

Run it, then the cell after it: after `c.radius = 10`, the area printed is stale.

1. Change the setter of `radius` so that it invalidates the cache: `cached_property` stores the value in the instance's dictionary, `self.__dict__`, under the name `"area"`. The next cell must then print `Computing the area...` twice, and an area of about `314.16` at the end.
2. Add a deleter to `radius`, which removes `_radius` and the cache: after `del c.radius`, `vars(c)` must be `{}`.

**Useful documentation**

- [`functools.cached_property`](https://docs.python.org/3/library/functools.html#functools.cached_property)
- [`dict.pop`](https://docs.python.org/3/library/stdtypes.html#dict.pop)
- [`property.deleter`](https://docs.python.org/3/library/functions.html#property.deleter)

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("The radius must be strictly positive")
    self._radius = value
    # Your code here

  @functools.cached_property
  def area(self):
    print("Computing the area...")
    return math.pi * self.radius**2

In [ ]:
c = Circle(5)
print(c.area)
print(c.area)
c.radius = 10
print(c.area)

### Additional hints

- `self.__dict__.pop("area", None)` removes the cached value. Without the `None`, the constructor would crash: `__init__` goes through the setter while nothing is cached yet, and `pop` would raise a `KeyError`.
- The deleter is written under `@radius.deleter`, also with the name `radius`.

```python
@radius.setter
def radius(self, value):
  if value <= 0:
    raise ValueError("The radius must be strictly positive")
  self._radius = value
  self.__dict__.pop(...)


@radius.deleter
def radius(self):
  del self._radius
  self.__dict__.pop(...)
```

### Solution

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("The radius must be strictly positive")
    self._radius = value
    self.__dict__.pop("area", None)

  @radius.deleter
  def radius(self):
    del self._radius
    self.__dict__.pop("area", None)

  @functools.cached_property
  def area(self):
    print("Computing the area...")
    return math.pi * self.radius**2


c = Circle(5)
print(c.area)
print(c.area)
c.radius = 10
print(c.area)
del c.radius
print(vars(c))

`cached_property` requires the instances to have a `__dict__`: that's the default, unless the class defines `__slots__` without `"__dict__"`.

## What `@x.setter` Does

`@property` is only syntactic sugar, and so is `@radius.setter`. To see what it does, write a `Circle` class without any `@`: the getter is named `read`, the setter `write` (with the same validation as above), and the `radius` property is built in two assignments, first `property` with the getter alone, then that property's `setter` method.

Expected result: `Circle(5).radius` is `5`, `Circle.radius.fget.__name__` is `"read"` and `Circle.radius.fset.__name__` is `"write"`.

**Useful documentation**

- [`property`](https://docs.python.org/3/library/functions.html#property)
- [`property.setter`](https://docs.python.org/3/library/functions.html#property.setter)

In [ ]:
# Your code here

Does the `setter` method change the property, or return another one? Why, then, do the two methods have the same name when writing `@radius.setter`?

*Your answer*

### Additional hints

Both assignments go in the class body, after the two methods. To answer the question, compare with `is` a property and what its `setter` method returns.

```python
class Circle:
  def __init__(self, radius):
    self.radius = radius

  def read(self):
    return self._radius

  def write(self, value): ...

  radius = property(...)
  radius = radius.setter(...)
```

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  def read(self):
    return self._radius

  def write(self, value):
    if value <= 0:
      raise ValueError("The radius must be strictly positive")
    self._radius = value

  radius = property(read)
  radius = radius.setter(write)


print(Circle(5).radius)
print(Circle.radius.fget.__name__, Circle.radius.fset.__name__)

without_setter = property(Circle.read)
with_setter = without_setter.setter(Circle.write)
print(without_setter is with_setter, without_setter.fset)

`setter` doesn't change the property: it returns a new one, which also has the setter. So the result has to be stored under the name `radius`.

That's what `@radius.setter` does above `def radius`: the decorator's result takes the method's name. If the method is named `write`, `write` gets the new property, and `radius` stays without a setter.

## Going Further: A Reusable Descriptor

This exercise is optional.

A `Speaker` has a `volume` between 0 and 100 and a `bass` between -10 and 10. With properties, each attribute would need a getter and a setter. Instead, complete the `Between` descriptor, which validates any attribute between two bounds: add `__set_name__`, `__get__` and `__set__` to it.

Once the descriptor is written, the test cell must print `30 80`, an error message naming `volume`, then `30`: two speakers don't share their values, and a refused value changes nothing.

In [ ]:
class Between:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  # Your code here


class Speaker:
  volume = Between(0, 100)
  bass = Between(-10, 10)

  def __init__(self, volume, bass):
    self.volume = volume
    self.bass = bass

In [ ]:
living_room = Speaker(30, 2)
kitchen = Speaker(80, -5)
print(living_room.volume, kitchen.volume)

try:
  living_room.volume = 150
except ValueError as error:
  print(f"Error: {error}")

print(living_room.volume)

### Additional hints

- `__set_name__(self, owner, name)` is called when the class is created, once per attribute: it gives the attribute's name (`"volume"`).
- A single `Between` serves every instance of `Speaker`: the value doesn't go in `self`, but in `instance`, with `setattr` and `getattr`, under an internal name (`"_volume"`).
- Read on the class (`Speaker.volume`), `__get__` receives `None` for `instance`: it then returns the descriptor itself.

```python
class Between:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  def __set_name__(self, owner, name):
    self.private_name = ...

  def __get__(self, instance, owner=None):
    if instance is None:
      return self
    return getattr(...)

  def __set__(self, instance, value):
    if not ...:
      raise ValueError(...)
    setattr(...)
```

### Solution

In [ ]:
class Between:
  def __init__(self, minimum, maximum):
    self.minimum = minimum
    self.maximum = maximum

  def __set_name__(self, owner, name):
    self.private_name = "_" + name

  def __get__(self, instance, owner=None):
    if instance is None:
      return self
    return getattr(instance, self.private_name)

  def __set__(self, instance, value):
    if not self.minimum <= value <= self.maximum:
      raise ValueError(
        f"{self.private_name[1:]} must be between {self.minimum} and {self.maximum}"
      )
    setattr(instance, self.private_name, value)


class Speaker:
  volume = Between(0, 100)
  bass = Between(-10, 10)

  def __init__(self, volume, bass):
    self.volume = volume
    self.bass = bass


living_room = Speaker(30, 2)
kitchen = Speaker(80, -5)
print(living_room.volume, kitchen.volume)

try:
  living_room.volume = 150
except ValueError as error:
  print(f"Error: {error}")

print(living_room.volume)
print(vars(living_room))